In [1]:
import pubchempy as pcp

In [4]:
c = pcp.Compound.from_cid(440289)

In [6]:
c.canonical_smiles

'C1=CC=C(C(=C1)C(=O)O)NC2C(C(C(O2)COP(=O)(O)O)O)O'

In [7]:
import pandas as pd

metabolites_df = pd.read_csv('../data/NRRL_1/7_Annotation/chr.kegg.metabolites_backup.csv', dtype={'Formula/Composition': str}).fillna({'Formula/Composition': ''})

In [8]:
metabolites_df

,Metabolite ID,Name,Formula/Composition,Compartment
0,C00345,6-Phospho-D-gluconate,C6H13O10P,Cytosol
1,C00006,NADP+,C21H29N7O17P3,Cytosol
2,C00199,D-Ribulose 5-phosphate,C5H11O8P,Cytosol
3,C00011,CO2,CO2,Cytosol
4,C00005,NADPH,C21H30N7O17P3,Cytosol
...,...,...,...,...
1788,C00078_e,L-Tryptophan_e,test,e
1789,C00106_e,Uracil_e,test,e
1790,C00105_e,Uridine_e,test,e
1791,C00183_e,L-Valine_e,test,e


In [ ]:
from bioservices.kegg import KEGG
from Bio.KEGG.KGML.KGML_parser import read
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import pubchempy as pcp

starttime = datetime.now()

# 初始化KEGG服务
k = KEGG()

metabolites_df = metabolites_df.head(50)  # 选取前50行

metabolites_df['PubChem ID'] = ""
metabolites_df['Canonical SMILES'] = ""

# 遍历每一行（通过索引直接修改原DataFrame）
for idx in tqdm(metabolites_df.index, total=len(metabolites_df), desc="Adding PubChem IDs and SMILES"):
    metabolite_id = metabolites_df.loc[idx, 'Metabolite ID']
    try:
        # 获取KEGG数据并解析PATHWAY
        data = k.get(metabolite_id)
        if not data:  
            raise ValueError(f"没有找到代谢物 {metabolite_id} 的数据")
            continue
        dict_data = k.parse(data)
        # 'DBLINKS': {'PubChem': '5896', 'ChEBI': '29157'},
        dblinks = dict_data.get('DBLINKS', {})
        if not dblinks:
            raise ValueError(f"没有找到代谢物 {metabolite_id} 的DBLINKS")
            continue
        pubchem_id = dblinks.get('PubChem', '')
        if not pubchem_id:
            raise ValueError(f"没有找到代谢物 {metabolite_id} 的PubChem ID")
            continue
        # 直接更新原DataFrame的PubChem ID列
        metabolites_df.at[idx, 'PubChem ID'] = pubchem_id
        
        # 获取Canonical SMILES
        c = pcp.Compound.from_cid(pubchem_id)
        if canonical_smiles:
            metabolites_df.at[idx, 'Canonical SMILES'] = c.canonical_smiles
        else:
            raise ValueError(f"没有找到代谢物 {metabolite_id} 的Canonical SMILES")
            continue
        
    except Exception as e:
        print(f"处理代谢物 {metabolite_id} 时出错: {str(e)}")
        continue

# 保存更新后的数据
metabolites_df.to_csv("../data/NRRL_1/7_Annotation/chr.kegg.metabolites_backup_updated.csv", index=False)

endtime = datetime.now()
print(f"总耗时: {endtime - starttime}")